# 🧩 Mini-Lab: Fixtures and Golden Files

**Module: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Build** reusable pytest fixtures that supply shared test data to multiple tests
2. **Use** `tmp_path` to create temporary files and directories that auto-clean after each test
3. **Implement** golden-file comparisons that catch unexpected output changes

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Test Fixture | Reusable sample data and helpers (like pytest fixtures and tmp_path) that make tests repeatable, isolated, and easy to read, often using golden-output comparison to catch regressions. |
| Unit Test | A small assertion about one behavior. A unit test that cannot fail is theatre, it should catch a real bug when the behavior breaks. |
| Filesystem Paths | Use Path objects, not strings, to build file locations that work across OSes. Prefer relative paths anchored to your project (not your shell), since relative paths depend on the current working directory. |


## Why Fixtures and Golden Files?

In the previous mini-lab you wrote standalone tests with hard-coded inputs. That works for two or three tests, but as your test suite grows you'll notice:

1. **Repeated setup** — multiple tests need the same sample data, so you copy-paste it everywhere.
2. **Temporary files** — some functions read/write files; you need a safe scratch directory that cleans itself up.
3. **Output drift** — a function's output changes subtly after a refactor; you want to compare against a known-good "golden" snapshot.

Fixtures solve (1) and (2). Golden-file comparisons solve (3).

## Setup

In [1]:
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

print("ipytest ready.")

ipytest ready.


## The Code Under Test

We'll test a small function that takes a list of contact dicts and writes a formatted report to a file. This is exactly the kind of deterministic glue code from the testability map — no LLM involved.

In [3]:
from pathlib import Path


def write_contact_report(contacts: list[dict], output_path: Path) -> Path:
    """Write a formatted contact report to a file and return the path."""
    lines = ["# Contact Report", ""]
    for c in contacts:
        lines.append(f"- {c['name']} ({c['email']})")
    lines.append("")
    lines.append(f"Total: {len(contacts)} contacts")
    output_path.write_text("\n".join(lines))
    return output_path


print("write_contact_report() defined.")

write_contact_report() defined.


## Step 1 — A Reusable Fixture

> 🧩 **Fixtures and test data.** Watch for the same sample contacts to be injected by name into both tests.

A **fixture** is a function decorated with `@pytest.fixture`. Any test that names it as a parameter receives the fixture's return value automatically — no manual calling needed.

```python
@pytest.fixture
def sample_contacts():       # ← define once
    return [{...}, {...}]

def test_something(sample_contacts):   # ← inject by name
    assert len(sample_contacts) == 2
```

This eliminates copy-pasting the same list into every test.


In [4]:
%%ipytest -v

import pytest


@pytest.fixture
def sample_contacts():
    """Reusable test data — defined once, used by many tests."""
    return [
        {"name": "Alice Smith", "email": "alice@example.com"},
        {"name": "Bob Jones", "email": "bob@example.com"},
    ]


# ── Two tests sharing the same fixture ──────────────────────

def test_report_has_all_names(sample_contacts, tmp_path):
    """Every contact name appears in the report."""
    out = tmp_path / "report.md"
    write_contact_report(sample_contacts, out)
    text = out.read_text()
    for contact in sample_contacts:
        assert contact["name"] in text


def test_report_shows_correct_total(sample_contacts, tmp_path):
    """The total line matches the number of contacts."""
    out = tmp_path / "report.md"
    write_contact_report(sample_contacts, out)
    text = out.read_text()
    assert "Total: 2 contacts" in text

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_ed232d2137394a1ab4e522ea08db2dc7.py 

.

.

                                                     [100%]


======================================== 2 passed in 0.25s ========================================


### What just happened?

- `sample_contacts` — our custom fixture, injected by name into both tests.
- `tmp_path` — a **built-in pytest fixture** that gives each test its own temporary directory as a `pathlib.Path`. It is automatically cleaned up after the test finishes.

Neither test had to create or delete temporary files manually. Fixtures handled everything.

## Step 2 — `tmp_path` in Action

`tmp_path` is cross-platform (works on Mac, Windows, Linux) and gives each test a **unique** directory, so tests never interfere with each other.

Let's prove the isolation:

In [5]:
%%ipytest -v

collected_paths = []


def test_tmp_path_a(tmp_path):
    """Write a file in this test's temp dir."""
    f = tmp_path / "a.txt"
    f.write_text("from test a")
    collected_paths.append(tmp_path)
    assert f.exists()


def test_tmp_path_b(tmp_path):
    """This test gets a completely different temp dir."""
    f = tmp_path / "a.txt"          # same filename…
    assert not f.exists()            # …but it doesn't exist here
    collected_paths.append(tmp_path)


def test_paths_are_unique():
    """Confirm the two tests got different directories."""
    assert len(collected_paths) == 2
    assert collected_paths[0] != collected_paths[1]

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 3 items

t_ed232d2137394a1ab4e522ea08db2dc7.py 

.

.

.

                                                    [100%]



======================================== 3 passed in 0.13s ========================================


Each test got its own sandbox. File `a.txt` written in test A doesn't leak into test B. This is why `tmp_path` is the standard approach for testing file I/O.

## Step 3 — Golden-File Comparison

A **golden file** is a saved copy of the "known-good" output. Your test generates fresh output and compares it character-by-character against the golden file. If anything changes — even a missing newline — the test fails.

This is how you catch **output drift**: subtle changes introduced by refactoring that no individual `assert` would catch.

### The workflow

1. Run the function and save its output as the golden file (done once).
2. In the test, run the function again and compare against the golden file.
3. If the output changes intentionally, update the golden file.

In [6]:
%%ipytest -v

import pytest


# ── Our known-good "golden" output ────────────────────────
GOLDEN_REPORT = """# Contact Report

- Alice Smith (alice@example.com)
- Bob Jones (bob@example.com)

Total: 2 contacts"""


@pytest.fixture
def sample_contacts():
    return [
        {"name": "Alice Smith", "email": "alice@example.com"},
        {"name": "Bob Jones", "email": "bob@example.com"},
    ]


@pytest.fixture
def golden_path(tmp_path):
    """Write the golden file into the test's temp dir."""
    gf = tmp_path / "golden_report.md"
    gf.write_text(GOLDEN_REPORT)
    return gf


def test_output_matches_golden(sample_contacts, golden_path, tmp_path):
    """Generate a fresh report and compare it to the golden file."""
    actual_path = tmp_path / "actual_report.md"
    write_contact_report(sample_contacts, actual_path)

    actual = actual_path.read_text()
    expected = golden_path.read_text()

    assert actual == expected, (
        f"Output drift detected!\n"
        f"--- Expected ---\n{expected}\n"
        f"--- Actual ---\n{actual}"
    )

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_ed232d2137394a1ab4e522ea08db2dc7.py 

.

                                                      [100%]



======================================== 1 passed in 0.13s ========================================


The golden-file test passed — the function's output matches the known-good snapshot exactly.

Notice how fixtures compose: `golden_path` itself uses `tmp_path`. Pytest wires them together automatically.

## Step 4 — Break the Code, Watch the Golden Test Fail

As with the previous mini-lab: a test you've never seen fail is a test you can't trust. Let's introduce a subtle bug and confirm the golden comparison catches it.

In [7]:
def write_contact_report_buggy(contacts: list[dict], output_path: Path) -> Path:
    """Bug: uses semicolon instead of comma in the parenthetical."""
    lines = ["# Contact Report", ""]
    for c in contacts:
        lines.append(f"- {c['name']} ({c['email']})")
    lines.append("")
    lines.append(f"Total: {len(contacts)} contact(s)")  # ← subtle change
    output_path.write_text("\n".join(lines))
    return output_path


print("Buggy version defined — 'contacts' → 'contact(s)' in the total line.")

Buggy version defined — 'contacts' → 'contact(s)' in the total line.


In [8]:
%%ipytest -v


@pytest.fixture
def sample_contacts():
    return [
        {"name": "Alice Smith", "email": "alice@example.com"},
        {"name": "Bob Jones", "email": "bob@example.com"},
    ]


@pytest.fixture
def golden_path(tmp_path):
    gf = tmp_path / "golden_report.md"
    gf.write_text(GOLDEN_REPORT)
    return gf


def test_buggy_output_vs_golden(sample_contacts, golden_path, tmp_path):
    """This SHOULD FAIL — the buggy function produces different output."""
    actual_path = tmp_path / "actual_report.md"
    write_contact_report_buggy(sample_contacts, actual_path)

    actual = actual_path.read_text()
    expected = golden_path.read_text()

    assert actual == expected, (
        f"Output drift detected!\n"
        f"--- Expected ---\n{expected}\n"
        f"--- Actual ---\n{actual}"
    )

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_ed232d2137394a1ab4e522ea08db2dc7.py 

F

                                                      [100%]



============================================ FAILURES =============================================
___________________________________ test_buggy_output_vs_golden ___________________________________

sample_contacts = [{'name': 'Alice Smith', 'email': 'alice@example.com'}, {'name': 'Bob Jones', 'email': 'bob@example.com'}]
golden_path = WindowsPath('C:/Users/molak/AppData/Local/Temp/pytest-of-molak/pytest-148/test_buggy_output_vs_golden0/golden_report.md')
tmp_path = WindowsPath('C:/Users/molak/AppData/Local/Temp/pytest-of-molak/pytest-148/test_buggy_output_vs_golden0')

    def test_buggy_output_vs_golden(sample_contacts, golden_path, tmp_path):
        """This SHOULD FAIL — the buggy function produces different output."""
        actual_path = tmp_path / "actual_report.md"
        write_contact_report_buggy(sample_contacts, actual_path)
    
        actual = actual_path.read_text()
        expected = golden_path.read_text()
    
>       assert actual == expected, (
            f"O

The test failed and the diff shows exactly what changed: `contacts` → `contact(s)`. That's the power of golden-file testing — it catches any output drift, no matter how small.

## Quick Reference

| Tool | What it does | When to use it |
|------|-------------|----------------|
| `@pytest.fixture` | Supplies reusable data/resources to tests by name | Whenever 2+ tests need the same setup |
| `tmp_path` | Built-in fixture; gives each test a unique temp directory (`pathlib.Path`) | Any test that reads or writes files |
| Golden-file comparison | Compares current output against a saved known-good snapshot | Catching subtle output drift after refactors |

## 🎯 Summary

### Key Takeaways

1. **Fixtures eliminate duplication** — define shared test data once with `@pytest.fixture` and inject it by name into any test that needs it
2. **`tmp_path` gives safe, isolated scratch directories** — each test gets its own temporary folder as a cross-platform `pathlib.Path`, automatically cleaned up
3. **Fixtures compose** — one fixture can use another (e.g., `golden_path` uses `tmp_path`), and pytest wires the dependency chain automatically
4. **Golden-file tests catch output drift** — comparing fresh output against a known-good snapshot detects even single-character changes after a refactor